
# پروژه کلاس بندی ژانر موسیقی 
برای رسیدن به این هدف نیاز داریم که داده هایی زیادی در ژانر های متفاوت داشته باشیم دیتاست 

gtzan 

در ده کلاس (ژانر) و هر کدام از کلاس ها ۱۰۰ نمونه موسیقی به مدت ۳۰ ثانیه تهیه کرده است

## تقسیم هر موسیقی به بخش های کوچکتر

هر ترک این دیتاست در قالب موسیقی ۳۰ ثانیه وجود دارد و باید این ۳۰ ثانیه را به بخش های کوچکتری تقسیم کنیم تا بتوانیم الگو را در هر قسمت پیدا کنیم که به این فرآیند

Framing 

گفته می شود

برای اینکه بدانیم هر موسیقی را به چند قسمت چند ثانیه ای تقسیم کنیم نیاز به تکرار و آزمایش کردن داریم

زمانی هم که هر موسیقی را به قسمت های کوچکتر تقسیم می کنیم باید کاری کنیم که از 

data leakage 

جلوگیری شود 

برای این موضوع نیز باید هر قسمت موسیقی که تقسیم می شود همه قسمت های آن در بخش آموزش باید باشد یا حتما همشون در بخش آزمایش


در قسمت 


`make_features.py`

من این فیچرها را ساختم و از این برای مدل هم استفاده می کنم

## ویژگی هایی صدا
- MFCC (mel frequency cepstral coefficients)
- Spectral Centroid
- Band Energy Ratio
- BandWidth


### وارد کردن کتابخانه 

In [57]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.pipeline import make_pipeline, Pipeline

from sklearn.metrics import average_precision_score

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('/Users/pouniq/music_genre_classification/audio_features/gtzan_5s_features.csv')

In [22]:
mfcc_features = [
    "mfcc1_mean", "mfcc1_var",
    "mfcc2_mean", "mfcc2_var",
    "mfcc3_mean", "mfcc3_var",
    "mfcc4_mean", "mfcc4_var",
    "mfcc5_mean", "mfcc5_var",
    "mfcc6_mean", "mfcc6_var",
    "mfcc7_mean", "mfcc7_var",
    "mfcc8_mean", "mfcc8_var",
    "mfcc9_mean", "mfcc9_var",
    "mfcc10_mean", "mfcc10_var",
    "mfcc11_mean", "mfcc11_var",
    "mfcc12_mean", "mfcc12_var",
    "mfcc13_mean", "mfcc13_var",
    "mfcc14_mean", "mfcc14_var",
    "mfcc15_mean", "mfcc15_var",
    "mfcc16_mean", "mfcc16_var",
    "mfcc17_mean", "mfcc17_var",
    "mfcc18_mean", "mfcc18_var",
    "mfcc19_mean", "mfcc19_var",
    "mfcc20_mean", "mfcc20_var",
    "ber1_mean","ber1_var"
]

In [27]:
X = df.drop(columns=["file", "genre", "segment", "start_sec"])
y = df["genre"]
groups = df["file"]

In [36]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
model = make_pipeline(StandardScaler(), RandomForestClassifier())

scores = cross_validate(model, X, y, groups=groups, cv=cv,
                        scoring=["accuracy", "f1_macro"])

In [37]:
print(scores["test_f1_macro"].mean(), scores["test_f1_macro"].std())


0.7105568611460488 0.01956425258512277


In [30]:
for tr, te in cv.split(X, y, groups):
    assert set(groups.iloc[tr]).isdisjoint(groups.iloc[te])

## MODEL SELECTION
- SVC
- RF
- LOGISTIC REGRESSION
- NAIVE BAYES


In [54]:
models = {
    'dt': Pipeline(
        [
            ('model', SVC(
                probability=True,
                random_state= RANDOM_STATE
                
                ))
        ]
    ),
    
    'hgb': Pipeline(
        [
            ('model', HistGradientBoostingClassifier(
                random_state= RANDOM_STATE
            ))
        ]
    ),
    'rf': Pipeline(
        [
            ('model', RandomForestClassifier(
                random_state= RANDOM_STATE,

            ))
        ]
    ),
    'lg': Pipeline(
                [
            ('model', LogisticRegression(
                random_state= RANDOM_STATE

            ))
        ]
    )

}



In [ ]:
for name, m in models.items():

    pr_auc = []

    for tr_idx, te_idx in cv.split(X, y, groups=groups):

        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        m.fit(X_tr, y_tr)

        if hasattr(m, 'predict_proba'):
            pred_score = m.predict_proba(X_te)
        else:
            pred_score = m.decision_function(X_te)

        score = average_precision_score(
            y_te,
            pred_score,
            average='macro'
        )

        pr_auc.append(score)

    print(f"{name}: {np.mean(pr_auc):.4f} (+/- {np.std(pr_auc):.4f})")

/Users/pouniq/music_genre_classification/mgc/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/pouniq/music_genre_classification/mgc/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/pouniq/music_genre_classification/mgc/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/pouniq/music_genre_classification/mgc/lib/python3.14/site-packages/sklearn/

dt: 0.2711 (+/- 0.0119)


In [50]:
y = y.astype('category').cat.codes
y

0       0
1       0
2       0
3       0
4       0
       ..
5980    9
5981    9
5982    9
5983    9
5984    9
Length: 5985, dtype: int8